# 01 — Inversión gravimétrica 3D independiente

Primera inversión suave y reproducible con SimPEG. Se reservan 38 estaciones para validación ciega. La topografía se toma del DEM `data/processed/topography/topography_xyz_corregida.txt` y se complementa con las elevaciones de las estaciones.

In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from mantle3d.gravity import load_config, load_gravity, run_inversion, save_result

config = load_config(ROOT)
gravity_data = load_gravity(ROOT, config)
gravity_data.describe().round(3)

## Datos y separación de validación

In [ ]:
xyz = gravity_data[['x','y','z']].to_numpy()
values = gravity_data.value.to_numpy()
order = np.lexsort((xyz[:,1], xyz[:,0]))
holdout_preview = order[::config['validation']['holdout_stride']]
train_preview = np.setdiff1d(np.arange(len(values)), holdout_preview)
fig, ax = plt.subplots(figsize=(8,6))
sc=ax.scatter(xyz[train_preview,0],xyz[train_preview,1],c=values[train_preview],cmap='RdBu_r',s=35,label='Inversión')
ax.scatter(xyz[holdout_preview,0],xyz[holdout_preview,1],marker='s',facecolors='none',edgecolors='black',s=55,label='Validación')
ax.set(title='Anomalía gravimétrica y estaciones retenidas',xlabel='X (m)',ylabel='Y (m)')
ax.ticklabel_format(style='plain'); ax.set_aspect('equal'); ax.legend()
fig.colorbar(sc,ax=ax,label='Anomalía (mGal)'); plt.show()

## Convención de signo

Los datos de campo usan la convención gravimétrica tradicional: positivo hacia abajo. SimPEG usa un sistema cartesiano con `z` positivo hacia arriba y `gz` positivo hacia arriba. Por ello se invierte `d_simpeg = −d_campo`. No se cambia el signo del modelo después de la inversión; las predicciones se convierten de nuevo al signo de campo para compararlas y graficarlas.

In [ ]:
sign_to_simpeg=config['data']['observed_sign_to_simpeg']
sign_check=pd.DataFrame({'dato_campo_mGal':values[:5],'dato_simpeg_mGal':sign_to_simpeg*values[:5]})
sign_check

## Construcción y selección de la malla

La inversión reutiliza la malla común versionada en `outputs/common_mesh/` cuando ya existe. Si esos artefactos no están presentes, se construye una malla tensorial desde los 187 datos gravimétricos: celda central de **250 m**, aproximadamente 0.60 veces la separación mediana entre estaciones (418 m), núcleo de la campaña más 500 m, 6 celdas laterales crecientes por lado (factor 1.4), 22 celdas verticales centrales y 6 celdas verticales crecientes (factor 1.35). El techo se coloca 50 m sobre la elevación máxima y las celdas activas se definen bajo una topografía remuestreada desde `data/processed/topography/topography_xyz_corregida.txt` y las elevaciones de estaciones.

## Inversión

Se usa una malla tensorial con celda central de 250 m, pesos de sensibilidad, regularización sparse IRLS con normas `[0, 0.5, 0.5, 0.5]`, pesos por profundidad, pesos explícitos `alpha_s=0.01` y `alpha_x=alpha_y=alpha_z=0.5`, piso de incertidumbre de 0.70 mGal y contraste de densidad acotado durante la optimización y recortado al final entre **−500 y 500 kg/m³** (equivalente a **−0.5 y 0.5 g/cm³** en SimPEG).

In [ ]:
result = run_inversion(ROOT)
mesh = result['mesh']
print(f"Malla: {mesh.shape_cells}; celdas={mesh.n_cells:,}; activas={result['active'].sum():,}")
print(f"Nodos X: {mesh.nodes_x.min():.3f} a {mesh.nodes_x.max():.3f} m")
print(f"Nodos Y: {mesh.nodes_y.min():.3f} a {mesh.nodes_y.max():.3f} m")
print(f"Centros X: {mesh.cell_centers_x.min():.3f} a {mesh.cell_centers_x.max():.3f} m")
print(f"Centros Y: {mesh.cell_centers_y.min():.3f} a {mesh.cell_centers_y.max():.3f} m")
print(f"Malla común: {result['mesh_source']}; SHA-256={result['mesh_manifest']['mesh_sha256'][:16]}…")
print(f"Topografía: {result['topography_source']}")
print(f"Piso de incertidumbre: {result['uncertainty_floor']:.3f} mGal")
print(f"Normas IRLS: {config['regularization']['irls_norms']}")
print(f"Alphas regularización: s={config['regularization']['alpha_s']}, x={config['regularization']['alpha_x']}, y={config['regularization']['alpha_y']}, z={config['regularization']['alpha_z']}")
print(f"Depth weighting: {config['regularization']['depth_weighting']}")
print(f"Clip final a bounds: {config['inversion']['clip_final_to_bounds']}")
print(f"Rango de densidad: {result['model_active'].min():.3f} a {result['model_active'].max():.3f} g/cm³")

## Malla en el dominio del modelo

In [ ]:
mesh=result['mesh']; topo=result['topography']; xyz=result['xyz']
fig,axes=plt.subplots(1,2,figsize=(15,6),constrained_layout=True)
ax=axes[0]
for xline in mesh.nodes_x: ax.plot([xline,xline],[mesh.nodes_y.min(),mesh.nodes_y.max()],color='0.75',lw=.35)
for yline in mesh.nodes_y: ax.plot([mesh.nodes_x.min(),mesh.nodes_x.max()],[yline,yline],color='0.75',lw=.35)
ax.scatter(xyz[:,0],xyz[:,1],s=12,c='#E76F51',zorder=3,label='Estaciones')
ax.set(title='Vista en planta',xlabel='X (m)',ylabel='Y (m)',aspect='equal')
ax.ticklabel_format(style='plain'); ax.legend()
ax=axes[1]
for xline in mesh.nodes_x: ax.plot([xline,xline],[mesh.nodes_z.min(),mesh.nodes_z.max()],color='0.78',lw=.35)
for zline in mesh.nodes_z: ax.plot([mesh.nodes_x.min(),mesh.nodes_x.max()],[zline,zline],color='0.78',lw=.35)
y_levels=np.unique(topo[:,1]); y0=y_levels[np.argmin(abs(y_levels-np.median(xyz[:,1])))]
profile=topo[np.isclose(topo[:,1],y0)]; profile=profile[np.argsort(profile[:,0])]
ax.plot(profile[:,0],profile[:,2],color='#2A9D8F',lw=2,label='Topografía interpolada')
ax.scatter(xyz[:,0],xyz[:,2],s=10,c='#E76F51',alpha=.7,label='Elevaciones medidas')
ax.set(title='Sección X–Z del dominio',xlabel='X (m)',ylabel='Elevación (m)')
ax.ticklabel_format(style='plain'); ax.legend()
plt.show()

### Persistencia de la malla común

La malla no se reconstruye para cada método. Se guarda exactamente en `outputs/common_mesh/tensor_mesh.json`, junto con la topografía y máscara activa en `common_mesh_geometry.npz`. El manifiesto contiene hashes SHA-256. Las futuras inversiones magnética y MT deben cargar estos archivos y verificar sus hashes.

In [ ]:
pd.Series(result['mesh_manifest'],name='malla_común').to_frame()

## Ajuste y validación ciega

In [ ]:
train=result['train']; holdout=result['holdout']; observed=result['observed_field']
p_train=result['predicted_train_field']; p_hold=result['predicted_holdout_field']
rmse_train=np.sqrt(np.mean((p_train-observed[train])**2))
rmse_hold=np.sqrt(np.mean((p_hold-observed[holdout])**2))
corr_hold=np.corrcoef(p_hold,observed[holdout])[0,1]
metrics=pd.DataFrame({'conjunto':['Inversión','Validación ciega'],'n':[len(train),len(holdout)],'RMSE_mGal':[rmse_train,rmse_hold],'correlación':[np.corrcoef(p_train,observed[train])[0,1],corr_hold]})
display(metrics.round(3))
fig,ax=plt.subplots(figsize=(7,6),constrained_layout=True)
all_obs=np.r_[observed[train],observed[holdout]]; all_pred=np.r_[p_train,p_hold]
lim=[min(all_obs.min(),all_pred.min()),max(all_obs.max(),all_pred.max())]
ax.scatter(observed[train],p_train,c='#2A9D8F',edgecolor='white',s=42,label='Inversión')
ax.scatter(observed[holdout],p_hold,marker='s',facecolors='none',edgecolors='#C44536',s=64,linewidths=1.5,label='Validación ciega')
ax.plot(lim,lim,'k--',lw=1,label='1:1')
ax.set(title='Predicho vs observado',xlabel='Observado (mGal)',ylabel='Predicho (mGal)')
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal',adjustable='box'); ax.grid(True,alpha=.25); ax.legend()
fig_dir=ROOT/'outputs/figures'; fig_dir.mkdir(parents=True,exist_ok=True)
fig.savefig(fig_dir/'gravity_predicho_vs_observado.png',dpi=300)
fig.savefig(fig_dir/'gravity_predicho_vs_observado.svg')
plt.show()

## Diagnóstico de convergencia

In [ ]:
history=result['history']; iterations=np.arange(1,len(history['phi_d'])+1)
normalized_residual=(p_train-observed[train])/result['uncertainty_floor']
target=len(train)*config['inversion']['target_chi_factor']
lower=config['inversion']['density_lower_gcc']; upper=config['inversion']['density_upper_gcc']
at_bounds=np.mean(np.isclose(result['model_active'],lower,atol=1e-5)|np.isclose(result['model_active'],upper,atol=1e-5))*100
diagnostics=pd.DataFrame({
 'métrica':['Iteraciones','phi_d inicial','phi_d final','phi_d / n_d','Objetivo phi_d','Residuo normalizado medio','Desv. residuo normalizado','|residuo| ≤ 2σ','Celdas en límites','RMSE validación','Correlación validación'],
 'valor':[len(iterations),history['phi_d'][0],history['phi_d'][-1],history['phi_d'][-1]/len(train),target,normalized_residual.mean(),normalized_residual.std(),np.mean(abs(normalized_residual)<=2)*100,at_bounds,rmse_hold,corr_hold]
})
display(diagnostics.round(3))
converged=(history['phi_d'][-1]<=target and history['phi_d'][-1]<history['phi_d'][0])
print('CONVERGIÓ al objetivo estadístico.' if converged else 'NO alcanzó el objetivo estadístico.')
fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
axes[0].semilogy(iterations,history['phi_d'],'o-',label='phi_d')
axes[0].axhline(target,color='k',ls='--',label='Objetivo')
axes[0].set(xlabel='Iteración',ylabel='Misfit',title='Convergencia del ajuste'); axes[0].legend()
axes[1].semilogy(iterations,history['beta'],'o-',color='#7161A8')
axes[1].set(xlabel='Iteración',ylabel='Beta',title='Enfriamiento de regularización')
plt.show()

## Secciones del modelo recuperado

In [ ]:
model=result['model_full']; finite=np.abs(model[np.isfinite(model)])
clim=float(np.quantile(finite,0.98))
active_z=mesh.cell_centers[result['active'],2]
z_ind=int(np.argmin(np.abs(mesh.cell_centers_z-np.nanmax(active_z))))
y_levels=np.unique(result['topography'][:,1]); y0=y_levels[np.argmin(abs(y_levels-np.median(result['xyz'][:,1])))]
profile=result['topography'][np.isclose(result['topography'][:,1],y0)]; profile=profile[np.argsort(profile[:,0])]
zmin=float(np.nanmin(active_z)); zmax=float(max(np.nanmax(result['topography'][:,2]),np.nanmax(active_z)))
fig=plt.figure(figsize=(15,5.5),constrained_layout=True)
gs=fig.add_gridspec(1,3,width_ratios=[1,1,0.045])
axes=[fig.add_subplot(gs[0,0]),fig.add_subplot(gs[0,1])]
cax=fig.add_subplot(gs[0,2])
mesh.plot_slice(model,normal='Z',ind=z_ind,ax=axes[0],grid=False,pcolor_opts={'cmap':'RdBu_r','vmin':-clim,'vmax':clim})
axes[0].set_title(f'Sección horizontal z={mesh.cell_centers_z[z_ind]:.0f} m')
mesh.plot_slice(model,normal='Y',ind=mesh.shape_cells[1]//2,ax=axes[1],grid=False,pcolor_opts={'cmap':'RdBu_r','vmin':-clim,'vmax':clim})
axes[1].plot(profile[:,0],profile[:,2],color='k',lw=1.5,label='Topografía')
axes[1].set_ylim(zmin,zmax+250); axes[1].legend(loc='upper right')
axes[1].set_title('Sección vertical central')
color_scale=mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=-clim,vmax=clim),cmap='RdBu_r')
color_scale.set_array([])
fig.colorbar(color_scale,cax=cax,label='Contraste de densidad (g/cm³)')
plt.show()

## Exportación y dictamen

In [ ]:
output=save_result(result,ROOT)
print(f'Resultado guardado en: {output.relative_to(ROOT)}')
print(f'Modelo 3D NumPy: {output.with_name(output.stem + "_model_3d.npy").relative_to(ROOT)}')
print(f'Modelo 3D VTK: {output.with_name(output.stem + "_model_3d.vtk").relative_to(ROOT)}')
target = len(train) * config['inversion']['target_chi_factor']
print(f'Misfit final: {result["phi_d"]:.2f}; objetivo estadístico: {target:.2f}')
print(f'RMSE validación: {rmse_hold:.3f} mGal; correlación: {corr_hold:.3f}')

### Interpretación provisional

Esta corrida establece el modelo gravimétrico independiente de referencia. La validación ciega evalúa capacidad predictiva, pero no elimina la no unicidad. Antes de interpretar cuerpos geológicos se deben probar sensibilidad a malla, profundidad, incertidumbre, límites de densidad y topografía.